# fastuuid smoke test

Tests basic API surface of `fastuuid` — a fast Rust-backed UUID implementation for Python.

In [ ]:
# Example 1: uuid4 — random UUID generation.
from fastuuid import uuid4

u = uuid4()
print(f"uuid4() = {u}")
print(f"type    = {type(u).__name__}")
print(f"hex     = {u.hex}")
print(f"int     = {u.int}")
print(f"version = {u.version}")

# Each call yields a distinct value
u2 = uuid4()
assert u != u2, "uuid4() must produce distinct values"
assert u.version == 4
assert len(str(u)) == 36

In [ ]:
# Example 2: UUID constructor from string / hex / bytes / int.
from fastuuid import UUID

canonical = "12345678-1234-5678-1234-567812345678"
u_str = UUID(canonical)
u_hex = UUID(hex="12345678123456781234567812345678")
u_int = UUID(int=u_str.int)
u_bytes = UUID(bytes=u_str.bytes)

print(f"from string: {u_str}")
print(f"from hex   : {u_hex}")
print(f"from int   : {u_int}")
print(f"from bytes : {u_bytes}")

assert u_str == u_hex == u_int == u_bytes
assert str(u_str) == canonical

In [ ]:
# Example 3: uuid3 / uuid5 — namespace-based deterministic UUIDs.
# fastuuid exports no NAMESPACE_* constants; build the RFC 4122 DNS/URL
# namespaces by value. The Rust API takes the name as bytes.
import uuid as std_uuid

from fastuuid import UUID, uuid3, uuid5

NAMESPACE_DNS = UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")
NAMESPACE_URL = UUID("6ba7b811-9dad-11d1-80b4-00c04fd430c8")

name = b"example.com"
u3_a = uuid3(NAMESPACE_DNS, name)
u3_b = uuid3(NAMESPACE_DNS, name)
u5_a = uuid5(NAMESPACE_DNS, name)
u5_b = uuid5(NAMESPACE_URL, name)

print(f"uuid3(DNS, {name!r}) = {u3_a}  (version={u3_a.version})")
print(f"uuid5(DNS, {name!r}) = {u5_a}  (version={u5_a.version})")
print(f"uuid5(URL, {name!r}) = {u5_b}  (version={u5_b.version})")

assert u3_a == u3_b, "uuid3 must be deterministic"
assert u3_a.version == 3 and u5_a.version == 5
assert u5_a != u5_b, "different namespaces must yield different uuid5"
# Cross-check against the stdlib implementation (same RFC 4122 algorithm).
assert str(u3_a) == str(std_uuid.uuid3(std_uuid.NAMESPACE_DNS, "example.com"))
assert str(u5_a) == str(std_uuid.uuid5(std_uuid.NAMESPACE_DNS, "example.com"))


In [ ]:
# Example 4: round-trip + property accessors.
from fastuuid import UUID, uuid4

u = uuid4()
s = str(u)

# String round-trip
u_again = UUID(s)
assert u_again == u

# bytes round-trip
u_bytes_rt = UUID(bytes=u.bytes)
assert u_bytes_rt == u

# Check property surfaces exist and are coherent
print(f"u.urn     = {u.urn}")
print(f"u.variant = {u.variant}")
print(f"u.fields  = {u.fields}")
assert u.urn.startswith("urn:uuid:")
assert len(u.bytes) == 16

In [ ]:
# Example 5: bulk generation throughput + ordering.
import time
from fastuuid import uuid4

N = 10_000
t0 = time.perf_counter()
uuids = [uuid4() for _ in range(N)]
elapsed = time.perf_counter() - t0

print(f"Generated {N} UUIDs in {elapsed*1000:.2f} ms ({N/elapsed:,.0f} uuid/s)")

# Uniqueness check
unique = {str(u) for u in uuids}
assert len(unique) == N, f"expected {N} unique UUIDs, got {len(unique)}"

# Sorting works (UUID is comparable)
sorted_uuids = sorted(uuids)
assert sorted_uuids[0] <= sorted_uuids[-1]
print(f"\nfastuuid smoke test PASSED")